> Repository edition: run from the repository root using an environment with the dependencies in `requirements.txt`. Machine-specific environment paths have been replaced. Historical outputs are retained except environment checks; they are not evidence of a fresh rerun. See `docs/REPRODUCIBILITY.md`.


In [1]:
import sys
from pathlib import Path

print("Python executable:", sys.executable)
print("Working directory:", Path.cwd())


Python executable: c:\Users\AG\BeatrizGIT\Master\.venv\Scripts\python.exe
Working directory: c:\Users\AG\BeatrizGIT\Master\CapstoneProject


In [2]:
from pathlib import Path

feedback_path = Path("week_6")
inputs_path = feedback_path / "inputs.txt"
outputs_path = feedback_path / "outputs.txt"

print("Cumulative inputs found:", inputs_path.is_file())
print("Cumulative outputs found:", outputs_path.is_file())

Cumulative inputs found: True
Cumulative outputs found: True


This cell checks that the cumulative input and output files are available
in the week_6 folder. These files contain the evaluations from the six
completed rounds and will be combined with the initial observations to
prepare the seventh query round.

In [3]:
import ast
import re
import numpy as np
from pathlib import Path

initial_data_path = Path("initial_data")
feedback_path = Path("week_6")
completed_rounds = 6

def read_feedback_rounds(path):
    """Read cumulative rounds stored as lists or lists of NumPy arrays."""
    text = path.read_text(encoding="utf-8-sig").strip()
    text = re.sub(r"array\((\[[^\[\]]*\])\)", r"\1", text)

    tree = ast.parse(text, mode="exec")
    rounds = []

    for node in tree.body:
        if not isinstance(node, ast.Expr):
            raise ValueError(f"Unexpected file structure: {path}")
        rounds.append(ast.literal_eval(node.value))

    return rounds


input_rounds = read_feedback_rounds(feedback_path / "inputs.txt")
output_rounds = read_feedback_rounds(feedback_path / "outputs.txt")

assert len(input_rounds) == completed_rounds, (
    f"Expected {completed_rounds} input rounds, found {len(input_rounds)}."
)
assert len(output_rounds) == completed_rounds, (
    f"Expected {completed_rounds} output rounds, found {len(output_rounds)}."
)

assert all(len(r) == 8 for r in input_rounds), (
    "Each input round must contain eight functions."
)
assert all(len(r) == 8 for r in output_rounds), (
    "Each output round must contain eight functions."
)

expected_dimensions = {
    1: 2, 2: 2, 3: 3, 4: 4,
    5: 4, 6: 5, 7: 6, 8: 8
}

initial_data = {}
updated_data = {}

for function in range(1, 9):
    folder = initial_data_path / f"function_{function}"

    X_initial = np.load(
        folder / "initial_inputs.npy", allow_pickle=False
    )
    y_initial = np.load(
        folder / "initial_outputs.npy", allow_pickle=False
    ).reshape(-1)

    dimension = expected_dimensions[function]

    assert X_initial.ndim == 2
    assert X_initial.shape == (len(y_initial), dimension)

    X_new = np.asarray(
        [r[function - 1] for r in input_rounds], dtype=float
    )
    y_new = np.asarray(
        [r[function - 1] for r in output_rounds], dtype=float
    ).reshape(-1)

    assert X_new.shape == (completed_rounds, dimension)
    assert y_new.shape == (completed_rounds,)

    X = np.vstack([X_initial, X_new])
    y = np.concatenate([y_initial, y_new])

    assert np.isfinite(X).all(), f"Non-finite inputs: Function {function}"
    assert np.isfinite(y).all(), f"Non-finite outputs: Function {function}"
    assert ((X_new >= 0) & (X_new < 1)).all(), (
        f"Feedback inputs outside the permitted range: Function {function}"
    )

    initial_data[function] = {"X": X_initial, "y": y_initial}
    updated_data[function] = {"X": X, "y": y}

    print(
        f"Function {function}: "
        f"{len(y_initial)} → {len(y)} observations "
        f"| {dimension} variables"
    )

Function 1: 10 → 16 observations | 2 variables
Function 2: 10 → 16 observations | 2 variables
Function 3: 15 → 21 observations | 3 variables
Function 4: 30 → 36 observations | 4 variables
Function 5: 20 → 26 observations | 4 variables
Function 6: 20 → 26 observations | 5 variables
Function 7: 30 → 36 observations | 6 variables
Function 8: 40 → 46 observations | 8 variables


This cell loads the initial observations and combines them with the six
completed evaluation rounds. It checks the number of rounds, input
dimensions, numerical validity and permitted range of the submitted inputs.

The updated datasets contain 16, 16, 21, 36, 26, 26, 36 and 46 observations
for Functions 1–8, respectively. The totals differ because the supplied
initial datasets had different sizes. All observed outputs, including
negative values, are retained because every function is a maximisation task.

*The datasets are rebuilt from the initial files and one cumulative results
snapshot whenever this cell runs. This prevents repeated execution from
appending the same feedback again.*

In [4]:
import pandas as pd
from IPython.display import display

progress_rows = []

for function in range(1, 9):
    best_so_far = float(initial_data[function]["y"].max())

    for round_index in range(completed_rounds):
        observed_output = float(output_rounds[round_index][function - 1])
        improved = observed_output > best_so_far
        current_best = max(best_so_far, observed_output)

        progress_rows.append({
            "Function": function,
            "Round": round_index + 1,
            "Best before round": best_so_far,
            "Observed output": observed_output,
            "Improved": improved,
            "Improvement": current_best - best_so_far,
            "Current best": current_best
        })

        best_so_far = current_best

    # Check that the historical summary matches the updated dataset.
    assert best_so_far == float(updated_data[function]["y"].max())

progress_summary = pd.DataFrame(progress_rows)

round_6_summary = (
    progress_summary.loc[
        progress_summary["Round"] == completed_rounds
    ]
    .drop(columns="Round")
    .set_index("Function")
    .rename(columns={
        "Best before round": "Best before round 6",
        "Observed output": "Round 6 output",
        "Improved": "Round 6 improved"
    })
)

display(
    round_6_summary.style.format({
        "Best before round 6": "{:.6e}",
        "Round 6 output": "{:.6e}",
        "Improvement": "{:.6e}",
        "Current best": "{:.6e}"
    })
)

for round_number in range(1, completed_rounds + 1):
    improved_functions = progress_summary.loc[
        (progress_summary["Round"] == round_number)
        & progress_summary["Improved"],
        "Function"
    ].tolist()

    print(
        f"Functions improved in round {round_number}: "
        f"{improved_functions} | Count: {len(improved_functions)}"
    )

,Best before round 6,Round 6 output,Round 6 improved,Improvement,Current best
Function,,,,,
1,7.710875e-16,4.419488e-125,False,0.000000e+00,7.710875e-16
2,6.582395e-01,5.006244e-01,False,0.000000e+00,6.582395e-01
3,-3.096391e-02,-1.041767e-01,False,0.000000e+00,-3.096391e-02
4,4.098217e-01,3.156201e-01,False,0.000000e+00,4.098217e-01
5,4.779885e+03,6.584614e+03,True,1.804729e+03,6.584614e+03
6,-3.843809e-01,-2.424310e-01,True,1.419500e-01,-2.424310e-01
7,2.097864e+00,2.171300e+00,True,7.343605e-02,2.171300e+00
8,9.922955e+00,9.670310e+00,False,0.000000e+00,9.922955e+00


Functions improved in round 1: [2, 4, 5, 7, 8] | Count: 5
Functions improved in round 2: [6, 8] | Count: 2
Functions improved in round 3: [5, 8] | Count: 2
Functions improved in round 4: [6, 8] | Count: 2
Functions improved in round 5: [3, 4, 5, 8] | Count: 4
Functions improved in round 6: [5, 6, 7] | Count: 3


This cell evaluates each completed round against the best output available
before that round. An improvement is recorded only when the new observation
exceeds the previous best, since all eight functions are maximisation tasks.

The table summarises Round 6, while the printed lists show which functions
improved in each round. The best observed output is retained even when a
later query performs worse.

*This analysis measures optimisation progress rather than predictive
accuracy. A query that does not improve the current best can still provide
useful information about an unexplored region. Prediction errors and
uncertainty will be examined separately.*

Round 6 improved the best observed outputs for Functions 5, 6 and 7.
Function 7 improved for the first time since Round 1, while Function 8
did not improve after five consecutive rounds of improvement.

Three functions improved, compared with four in Round 5. This count alone
does not establish whether the strategy became better or worse: the
magnitude of each improvement and the information gained also matter.
Absolute improvements should be interpreted within each function because
their output scales differ.

*These results support reviewing performance separately for each function.
They do not yet establish that global or local search is consistently
superior.*


In [5]:
# Historical predictions recorded when selecting the Round 6 queries.
# Values are copied from the previous notebook's displayed results,
# so the diagnostic calculations use rounded prediction values.
round_6_predictions = {
    1: {"mean": -2.106466e-04, "std": 9.522384e-04},
    2: {"mean": 6.570358e-01, "std": 3.125545e-02},
    3: {"mean": -5.480886e-02, "std": 6.313081e-02},
    4: {"mean": 5.735155e-02, "std": 3.154412e-01},
    5: {
        "mean": 4.550247e+03,
        "std": 5.910056e+02,
        "auxiliary": 4.910054e+03
    },
    6: {"mean": -5.016286e-01, "std": 1.660142e-01},
    7: {
        "mean": 1.781922e+00,
        "std": 1.693531e-01,
        "auxiliary": 7.990466e-01
    },
    8: {"mean": 9.813935e+00, "std": 1.641871e-01}
}

round_6_expected_queries = {
    1: "0.123843-0.121237",
    2: "0.709248-0.306664",
    3: "0.045627-0.009123-0.525961",
    4: "0.424420-0.403134-0.390867-0.438795",
    5: "0.994128-0.987118-0.927371-0.967531",
    6: "0.432285-0.374372-0.559617-0.651650-0.005539",
    7: "0.000293-0.248999-0.558162-0.154628-0.355919-0.762975",
    8: (
        "0.086109-0.035213-0.037529-0.621722-"
        "0.929968-0.637726-0.311255-0.202704"
    )
}

review_rows = []

for function in range(1, 9):
    expected_query = np.array([
        float(value)
        for value in round_6_expected_queries[function].split("-")
    ])

    recorded_query = np.asarray(
        input_rounds[5][function - 1], dtype=float
    )

    # Ensure each historical prediction refers to the evaluated input.
    np.testing.assert_allclose(
        recorded_query,
        expected_query,
        rtol=0,
        atol=1e-9,
        err_msg=f"Round 6 query mismatch for Function {function}"
    )

    prediction = round_6_predictions[function]
    observed = float(output_rounds[5][function - 1])
    mean = prediction["mean"]
    std = prediction["std"]
    auxiliary = prediction.get("auxiliary", np.nan)
    error = observed - mean

    review_rows.append({
        "Function": function,
        "Predicted mean": mean,
        "Predicted std": std,
        "Observed output": observed,
        "Observed minus predicted": error,
        "Absolute error / predicted std": abs(error) / std,
        "GP absolute error": abs(error),
        "Auxiliary prediction": auxiliary,
        "Auxiliary absolute error": (
            abs(observed - auxiliary)
            if np.isfinite(auxiliary)
            else np.nan
        )
    })

prediction_review = pd.DataFrame(review_rows).set_index("Function")

formats = {
    column: "{:.6e}" for column in prediction_review.columns
}
formats["Absolute error / predicted std"] = "{:.2f}"

display(prediction_review.style.format(formats, na_rep="—"))

,Predicted mean,Predicted std,Observed output,Observed minus predicted,Absolute error / predicted std,GP absolute error,Auxiliary prediction,Auxiliary absolute error
Function,,,,,,,,
1,-2.106466e-04,9.522384e-04,4.419488e-125,2.106466e-04,0.22,2.106466e-04,—,—
2,6.570358e-01,3.125545e-02,5.006244e-01,-1.564114e-01,5.00,1.564114e-01,—,—
3,-5.480886e-02,6.313081e-02,-1.041767e-01,-4.936779e-02,0.78,4.936779e-02,—,—
4,5.735155e-02,3.154412e-01,3.156201e-01,2.582686e-01,0.82,2.582686e-01,—,—
5,4.550247e+03,5.910056e+02,6.584614e+03,2.034367e+03,3.44,2.034367e+03,4.910054e+03,1.674560e+03
6,-5.016286e-01,1.660142e-01,-2.424310e-01,2.591976e-01,1.56,2.591976e-01,—,—
7,1.781922e+00,1.693531e-01,2.171300e+00,3.893785e-01,2.30,3.893785e-01,7.990466e-01,1.372254e+00
8,9.813935e+00,1.641871e-01,9.670310e+00,-1.436253e-01,0.87,1.436253e-01,—,—


This cell compares the historical Round 6 predictions with the outputs
returned by the portal. It first verifies that each prediction corresponds
to the input actually evaluated.

Signed errors show whether the GP underestimated or overestimated the
output. Absolute errors are also divided by the predicted standard
deviation to assess how large each discrepancy was relative to the
model's reported uncertainty. For Functions 5 and 7, the auxiliary
predictions are evaluated alongside the GP predictions.

*These are prospective prediction checks: the predictions were recorded
before the new outputs were available. One result per function cannot
establish uncertainty calibration or determine which model is generally
more accurate. The historical predictions shown here are rounded.*

Round 6 highlights the distinction between optimisation progress and
predictive accuracy. Functions 5, 6 and 7 achieved new best observations,
although their GP models underestimated the returned outputs.

Function 2 showed the largest error relative to predicted uncertainty:
the observed output was approximately five predictive standard deviations
below the GP mean. This is a warning that the model was too confident at
this query and supports reassessing its fitted assumptions.

For Function 5, the GP underestimated the output by approximately 2,034,
or 3.44 predictive standard deviations. The auxiliary neural network had
a smaller absolute error of approximately 1,675, although it also
underestimated the result. For Function 7, the GP was closer to the
observed output than the auxiliary linear regression model.

Functions 1, 3, 4 and 8 had errors below one predictive standard deviation.
However, a small standardised error does not establish that a model is
well calibrated. For Function 1, predicting outputs near zero also provides
limited evidence about the ability to locate a substantially better region.

*These results motivate comparing model configurations using consistent
validation splits, particularly for Functions 2, 5 and 7. The latest
observations can now enter the training data, but the historical predictions
must remain unchanged. One evaluation per function is insufficient to
establish model superiority or justify a specific uncertainty adjustment.*

In [7]:
import warnings
from time import perf_counter

from sklearn.dummy import DummyRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel, Matern, WhiteKernel
)
from sklearn.model_selection import KFold

# A small, explicit grid of GP configurations.
gp_configs = {}

for nu in [2.5, 1.5]:
    for label, bounds in [
        ("original", (0.02, 2.0)),
        ("wider", (0.02, 10.0)),
        ("lower_and_wider", (0.005, 10.0))
    ]:
        name = f"Matern_{nu}_{label}"
        gp_configs[name] = {
            "nu": nu,
            "length_scale_bounds": bounds
        }


def make_gp(dimension, config, restarts=3):
    """Create a fresh GP for each training fold."""
    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * dimension,
            length_scale_bounds=config["length_scale_bounds"],
            nu=config["nu"]
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    return GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=restarts,
        random_state=42
    )


cv_rows = []
cv_fold_rows = []
cv_warning_log = []
cv_splits = {}

for function in range(1, 9):
    X = updated_data[function]["X"]
    y = updated_data[function]["y"]

    # Reuse exactly the same folds for all models of this function.
    splitter = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_splits[function] = list(splitter.split(X))

    models_to_compare = ["Training mean"] + list(gp_configs)

    for model_name in models_to_compare:
        start = perf_counter()
        train_errors = []
        validation_errors = []
        warning_count = 0

        for fold, (train_idx, val_idx) in enumerate(
            cv_splits[function], start=1
        ):
            if model_name == "Training mean":
                model = DummyRegressor(strategy="mean")
            else:
                model = make_gp(
                    X.shape[1], gp_configs[model_name]
                )

            # Capture convergence warnings without losing their details.
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always", ConvergenceWarning)
                model.fit(X[train_idx], y[train_idx])

            for warning in caught:
                cv_warning_log.append({
                    "Function": function,
                    "Model": model_name,
                    "Fold": fold,
                    "Category": warning.category.__name__,
                    "Message": str(warning.message)
                })
                if issubclass(warning.category, ConvergenceWarning):
                    warning_count += 1

            train_prediction = model.predict(X[train_idx])
            val_prediction = model.predict(X[val_idx])

            train_rmse = float(np.sqrt(
                np.mean((y[train_idx] - train_prediction) ** 2)
            ))
            val_rmse = float(np.sqrt(
                np.mean((y[val_idx] - val_prediction) ** 2)
            ))

            train_errors.append(train_rmse)
            validation_errors.append(val_rmse)

            cv_fold_rows.append({
                "Function": function,
                "Model": model_name,
                "Fold": fold,
                "Training RMSE": train_rmse,
                "Validation RMSE": val_rmse
            })

        cv_rows.append({
            "Function": function,
            "Model": model_name,
            "Mean training RMSE": np.mean(train_errors),
            "Mean CV RMSE": np.mean(validation_errors),
            "CV RMSE standard deviation": np.std(
                validation_errors, ddof=1
            ),
            "Convergence warnings": warning_count,
            "CV seconds": perf_counter() - start
        })

    print(f"Function {function}: cross-validation completed.")

gp_cv_results = pd.DataFrame(cv_rows)
gp_cv_fold_results = pd.DataFrame(cv_fold_rows)
gp_cv_warnings = pd.DataFrame(cv_warning_log)

gp_cv_results = gp_cv_results.sort_values(
    ["Function", "Mean CV RMSE"]
).reset_index(drop=True)

display(
    gp_cv_results.style.format({
        "Mean training RMSE": "{:.6e}",
        "Mean CV RMSE": "{:.6e}",
        "CV RMSE standard deviation": "{:.6e}",
        "CV seconds": "{:.2f}"
    })
)

Function 1: cross-validation completed.
Function 2: cross-validation completed.
Function 3: cross-validation completed.
Function 4: cross-validation completed.
Function 5: cross-validation completed.
Function 6: cross-validation completed.
Function 7: cross-validation completed.
Function 8: cross-validation completed.


,Function,Model,Mean training RMSE,Mean CV RMSE,CV RMSE standard deviation,Convergence warnings,CV seconds
0,1,Training mean,7.063947e-04,7.669330e-04,7.325198e-04,0,0.29
1,1,Matern_1.5_original,3.492307e-10,8.911499e-04,6.774462e-04,3,0.61
2,1,Matern_1.5_wider,3.502449e-10,8.929550e-04,6.776070e-04,3,0.68
3,1,Matern_1.5_lower_and_wider,7.130573e-10,9.027173e-04,6.641625e-04,2,0.45
4,1,Matern_2.5_wider,1.372259e-11,9.213283e-04,6.716444e-04,4,0.68
5,1,Matern_2.5_original,3.619130e-10,9.223259e-04,6.722934e-04,3,0.59
6,1,Matern_2.5_lower_and_wider,7.152019e-10,9.290873e-04,6.604213e-04,2,0.69
7,2,Matern_1.5_wider,1.797931e-02,1.994278e-01,4.897028e-02,2,0.80
8,2,Matern_1.5_original,1.592976e-02,2.003006e-01,5.086679e-02,2,0.71
9,2,Matern_1.5_lower_and_wider,3.473819e-07,2.070962e-01,5.733531e-02,2,0.52


This cell performs a small grid search over six Gaussian Process
configurations for each function. It compares two Matérn smoothness
settings (nu = 1.5 and 2.5) and three length-scale bound intervals.
The grid includes the previous configurations and an alternative that
allows shorter length scales.

Each configuration is evaluated using the same three cross-validation
folds within each function. Output normalisation and kernel fitting use
only the training portion of each fold. A training-mean baseline provides
a reference for assessing whether the GP adds predictive value.

The outer grid chooses the kernel smoothness and permitted length-scale
bounds. Within each fit, the GP optimiser estimates the individual length
scales, signal amplitude and noise level by maximising the log-marginal
likelihood. Three optimiser restarts keep the comparison computationally
manageable.

The table reports training RMSE, mean validation RMSE, variability across
folds and runtime. Lower validation RMSE is preferable, but small
differences must be considered alongside fold variability. Convergence
warnings are retained in gp_cv_warnings for inspection.

*Module 18 connection: this experiment applies a limited grid search to
balance search coverage, computational cost and generalisation. These
scores support configuration selection; they are not an independent
estimate of the selected model's future performance. The observations
were collected adaptively, so random-fold validation may not represent
performance in unexplored regions. RMSE also does not directly assess
uncertainty calibration or optimisation success.*

The grid search does not identify one configuration that is best for all
functions. Matérn nu = 1.5 achieves the lowest GP validation RMSE for
Functions 1–5, while nu = 2.5 remains preferable for Functions 6–8.

For Function 1, the training-mean baseline outperforms every tested GP.
For Function 3, the best GP is effectively tied with the baseline.
These results limit confidence in the predictive guidance available for
these functions. A GP is retained as a provisional search model because
it provides input-dependent predictions and uncertainty, but this does
not establish that its acquisition strategy will outperform random search.

Function 4 shows a lower validation error with nu = 1.5. For Function 5,
the difference between the two original-bound kernels is small relative
to fold variability. Its selected configuration should therefore be
treated as provisional.

Functions 6 and 7 retain their previous GP configurations. Function 8
also retains Matérn nu = 2.5 with bounds (0.02, 10), because additionally
lowering the lower bound produces effectively identical validation error.

*Selection follows the lowest observed GP validation RMSE, with continuity
preferred for the numerical tie in Function 8. Small differences between
scores are not evidence of statistically significant improvement. Near-zero
training errors alone do not prove overfitting, and validation RMSE does
not establish whether predictive uncertainty is well calibrated.*

In [8]:
# Store the chosen configuration for each function.
round_7_gp_config_names = {
    1: "Matern_1.5_original",
    2: "Matern_1.5_wider",
    3: "Matern_1.5_original",
    4: "Matern_1.5_wider",
    5: "Matern_1.5_original",
    6: "Matern_2.5_original",
    7: "Matern_2.5_original",
    8: "Matern_2.5_wider"
}

round_7_gp_models = {}
round_7_fit_warning_rows = []
fit_rows = []

for function, config_name in round_7_gp_config_names.items():
    X = updated_data[function]["X"]
    y = updated_data[function]["y"]
    config = gp_configs[config_name]

    # Keep the optimiser budget consistent with the CV experiment.
    model = make_gp(X.shape[1], config, restarts=3)

    start = perf_counter()

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(X, y)

    fit_seconds = perf_counter() - start
    round_7_gp_models[function] = model

    for warning in caught:
        round_7_fit_warning_rows.append({
            "Function": function,
            "Category": warning.category.__name__,
            "Message": str(warning.message)
        })

    selected_cv = gp_cv_results.loc[
        (gp_cv_results["Function"] == function)
        & (gp_cv_results["Model"] == config_name)
    ].iloc[0]

    fit_rows.append({
        "Function": function,
        "Observations": len(y),
        "Selected configuration": config_name,
        "Selection CV RMSE": selected_cv["Mean CV RMSE"],
        "CV RMSE standard deviation": (
            selected_cv["CV RMSE standard deviation"]
        ),
        "Fitted kernel": str(model.kernel_),
        "Log-marginal likelihood": (
            model.log_marginal_likelihood_value_
        ),
        "Fit seconds": fit_seconds
    })

round_7_fit_summary = pd.DataFrame(fit_rows).set_index("Function")
round_7_fit_warnings = pd.DataFrame(round_7_fit_warning_rows)

with pd.option_context("display.max_colwidth", None):
    display(
        round_7_fit_summary.style.format({
            "Selection CV RMSE": "{:.6e}",
            "CV RMSE standard deviation": "{:.6e}",
            "Log-marginal likelihood": "{:.6f}",
            "Fit seconds": "{:.3f}"
        })
    )

if not round_7_fit_warnings.empty:
    print("Warnings from the full-data fits:")
    display(
        round_7_fit_warnings.drop_duplicates().reset_index(drop=True)
    )
else:
    print("No warnings from the full-data fits.")

,Observations,Selected configuration,Selection CV RMSE,CV RMSE standard deviation,Fitted kernel,Log-marginal likelihood,Fit seconds
Function,,,,,,,
1,16,Matern_1.5_original,8.911499e-04,6.774462e-04,"1.06**2 * Matern(length_scale=[0.02, 2], nu=1.5) + WhiteKernel(noise_level=6.85e-08)",-21.341437,0.528
2,16,Matern_1.5_wider,1.994278e-01,4.897028e-02,"0.963**2 * Matern(length_scale=[0.0876, 10], nu=1.5) + WhiteKernel(noise_level=0.0672)",-15.877305,0.292
3,21,Matern_1.5_original,7.463068e-02,3.651134e-02,"1.32**2 * Matern(length_scale=[0.637, 2, 0.178], nu=1.5) + WhiteKernel(noise_level=0.00379)",-24.682851,0.280
4,36,Matern_1.5_wider,1.909213e+00,6.754003e-01,"4.89**2 * Matern(length_scale=[3.87, 3.55, 3.54, 3.69], nu=1.5) + WhiteKernel(noise_level=1.44e-05)",-4.398298,0.392
5,26,Matern_1.5_original,7.406537e+02,5.524297e+02,"2.1**2 * Matern(length_scale=[2, 2, 2, 1.77], nu=1.5) + WhiteKernel(noise_level=5.46e-10)",-12.796145,0.270
6,26,Matern_2.5_original,3.324422e-01,2.778027e-03,"1.52**2 * Matern(length_scale=[0.789, 1.12, 1.52, 0.852, 1.52], nu=2.5) + WhiteKernel(noise_level=0.0152)",-22.725028,0.345
7,36,Matern_2.5_original,4.176976e-01,5.108223e-02,"0.802**2 * Matern(length_scale=[2, 2, 1.35, 0.323, 2, 0.0352], nu=2.5) + WhiteKernel(noise_level=4.47e-09)",-32.021638,0.422
8,46,Matern_2.5_wider,1.717851e-01,2.592834e-02,"2.95**2 * Matern(length_scale=[2.97, 4.57, 2.38, 5.07, 10, 4.86, 3.07, 10], nu=2.5) + WhiteKernel(noise_level=1e-10)",7.158847,1.214


Warnings from the full-data fits:


,Function,Category,Message
0,1,ConvergenceWarning,The optimal value found for dimension 0 of par...
1,1,ConvergenceWarning,The optimal value found for dimension 1 of par...
2,2,ConvergenceWarning,The optimal value found for dimension 1 of par...
3,3,ConvergenceWarning,The optimal value found for dimension 1 of par...
4,5,ConvergenceWarning,The optimal value found for dimension 0 of par...
5,5,ConvergenceWarning,The optimal value found for dimension 1 of par...
6,5,ConvergenceWarning,The optimal value found for dimension 2 of par...
7,7,ConvergenceWarning,The optimal value found for dimension 0 of par...
8,7,ConvergenceWarning,The optimal value found for dimension 1 of par...
9,7,ConvergenceWarning,The optimal value found for dimension 4 of par...


This cell records the selected GP configurations and refits each model
using all observations available after six completed rounds. The fitted
models are stored in round_7_gp_models for subsequent candidate evaluation.

The optimiser restart budget remains the same as in cross-validation.
Kernel parameters are re-estimated on the complete dataset, so they may
differ from those fitted within individual validation folds.

The table preserves the configuration-selection scores alongside the
full-data fitted kernels and runtimes. Any fitting warnings are displayed
and retained for review.

*The validation scores belong to the earlier fold-based experiment; they
are not new performance measurements of the full-data models. No seventh-
round query has been selected or evaluated by this cell.*

The full-data fits retain several boundary constraints. Function 1 reaches
both a lower and an upper length-scale bound, reinforcing the caution
suggested by its weak validation performance.

For Function 5, three length scales reach the upper bound of 2. The bounds
are retained because the wider alternatives produced higher validation
errors. A boundary warning alone is insufficient justification for changing
the selected configuration.

Function 7 shows a substantial change in its fitted length scales,
including a short scale for x6. This suggests that the current surrogate
is sensitive to small changes in that coordinate, but the changing fit
also cautions against interpreting length scales as stable feature
importance estimates.

Function 8 reaches the upper bounds for x5 and x8 and the lower noise
bound. These estimates describe the fitted model and do not establish
that those inputs are irrelevant or that evaluations are noise-free.

*Kernel estimates depend on the available data, model assumptions and
optimisation procedure. Log-marginal likelihood values should not be used
to rank different black-box functions. The cross-validation results remain
the basis for the configuration choices in this round.*

In [9]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression

auxiliary_estimators = {
    5: (
        "NN (8,)",
        MLPRegressor(
            hidden_layer_sizes=(8,),
            activation="tanh",
            solver="lbfgs",
            alpha=0.1,
            max_iter=2000,
            random_state=42
        )
    ),
    7: ("Linear regression", LinearRegression())
}

round_7_auxiliary_models = {}
round_7_auxiliary_names = {}
auxiliary_fit_rows = []
auxiliary_warning_rows = []

for function, (name, estimator) in auxiliary_estimators.items():
    X = updated_data[function]["X"]
    y = updated_data[function]["y"]

    model = TransformedTargetRegressor(
        regressor=Pipeline([
            ("input_scaler", StandardScaler()),
            ("model", estimator)
        ]),
        transformer=StandardScaler()
    )

    start = perf_counter()

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(X, y)

    fit_seconds = perf_counter() - start

    round_7_auxiliary_models[function] = model
    round_7_auxiliary_names[function] = name

    for warning in caught:
        auxiliary_warning_rows.append({
            "Function": function,
            "Category": warning.category.__name__,
            "Message": str(warning.message)
        })

    auxiliary_fit_rows.append({
        "Function": function,
        "Observations": len(y),
        "Auxiliary model": name,
        "Fit seconds": fit_seconds
    })

round_7_auxiliary_summary = (
    pd.DataFrame(auxiliary_fit_rows).set_index("Function")
)
round_7_auxiliary_warnings = pd.DataFrame(auxiliary_warning_rows)

display(
    round_7_auxiliary_summary.style.format({
        "Fit seconds": "{:.3f}"
    })
)

for warning in auxiliary_warning_rows:
    print(
        f"Function {warning['Function']} — "
        f"{warning['Category']}: {warning['Message']}"
    )

,Observations,Auxiliary model,Fit seconds
Function,,,
5,26,"NN (8,)",0.249
7,36,Linear regression,0.026


This cell refits the existing auxiliary models using all observations
available after Round 6: a neural network with one hidden layer of eight
units for Function 5 and linear regression for Function 7.

Inputs and targets are standardised during fitting. Predictions are
automatically returned in the original output units. The auxiliary
configurations remain unchanged, keeping this round's tuning experiment
focused on the Gaussian Process models.

In Round 6, the neural network predicted Function 5 more accurately than
the GP, while linear regression predicted Function 7 less accurately.
These individual outcomes do not establish general model superiority.
Both auxiliary models are retained to monitor agreement and disagreement
with the GP at proposed query points.

*These fits do not constitute a new validation experiment. Auxiliary
predictions will be reported separately and will not be added to the
GP-based UCB score.*

In [10]:
from scipy.spatial import cKDTree

beta_round_7 = 2.0

round_7_candidate_proposals = {}
candidate_rows = []
search_time_rows = []


def predict_in_batches(model, candidates, batch_size=5000):
    """Evaluate candidates without a large prediction matrix."""
    means = []
    standard_deviations = []

    for start in range(0, len(candidates), batch_size):
        batch = candidates[start:start + batch_size]
        mean, std = model.predict(batch, return_std=True)
        means.append(mean)
        standard_deviations.append(std)

    return np.concatenate(means), np.concatenate(standard_deviations)


for function in range(1, 9):
    start = perf_counter()

    X = updated_data[function]["X"]
    y = updated_data[function]["y"]
    model = round_7_gp_models[function]

    dimension = X.shape[1]
    best_input = X[np.argmax(y)]
    current_best = float(y.max())

    rng = np.random.default_rng(42 + function)
    observation_tree = cKDTree(X)

    candidate_pools = {
        "Global": rng.uniform(
            0.0, 0.999999, size=(100000, dimension)
        )
    }

    for radius in [0.10, 0.03]:
        lower = np.maximum(0.0, best_input - radius)
        upper = np.minimum(0.999999, best_input + radius)

        candidate_pools[f"Local {radius:.2f}"] = rng.uniform(
            lower, upper, size=(10000, dimension)
        )

    round_7_candidate_proposals[function] = {}

    for search_name, candidates in candidate_pools.items():
        # Evaluate the exact coordinates that could be submitted.
        candidates = np.unique(np.round(candidates, 6), axis=0)

        nearest_distance, _ = observation_tree.query(candidates)
        keep = nearest_distance > 1e-6

        candidates = candidates[keep]
        nearest_distance = nearest_distance[keep]

        if len(candidates) == 0:
            raise ValueError(
                f"No unevaluated candidates for Function {function}, "
                f"{search_name}."
            )

        mean, std = predict_in_batches(model, candidates)
        ucb = mean + beta_round_7 * std

        assert np.isfinite(mean).all()
        assert np.isfinite(std).all()
        assert np.isfinite(ucb).all()

        index = int(np.argmax(ucb))
        query = candidates[index].copy()

        auxiliary_prediction = np.nan
        if function in round_7_auxiliary_models:
            auxiliary_prediction = float(
                round_7_auxiliary_models[function]
                .predict(query.reshape(1, -1))[0]
            )

        proposal = {
            "query": query,
            "mean": float(mean[index]),
            "std": float(std[index]),
            "ucb": float(ucb[index]),
            "nearest_distance": float(nearest_distance[index]),
            "auxiliary_prediction": auxiliary_prediction
        }

        round_7_candidate_proposals[function][search_name] = proposal

        candidate_rows.append({
            "Function": function,
            "Search": search_name,
            "Query": "-".join(f"{value:.6f}" for value in query),
            "Current best": current_best,
            "GP mean": proposal["mean"],
            "GP std": proposal["std"],
            "UCB": proposal["ucb"],
            "Nearest observation distance": proposal["nearest_distance"],
            "Auxiliary prediction": auxiliary_prediction
        })

    search_time_rows.append({
        "Function": function,
        "Candidate search seconds": perf_counter() - start
    })

round_7_candidate_comparison = pd.DataFrame(candidate_rows)
round_7_search_times = (
    pd.DataFrame(search_time_rows).set_index("Function")
)

# Keep Function and Search as ordinary columns for easier copying.
display(
    round_7_candidate_comparison.style
    .hide(axis="index")
    .format({
        "Current best": "{:.6e}",
        "GP mean": "{:.6e}",
        "GP std": "{:.6e}",
        "UCB": "{:.6e}",
        "Nearest observation distance": "{:.6f}",
        "Auxiliary prediction": "{:.6e}"
    }, na_rep="—")
)

display(
    round_7_search_times.style.format({
        "Candidate search seconds": "{:.3f}"
    })
)

Function,Search,Query,Current best,GP mean,GP std,UCB,Nearest observation distance,Auxiliary prediction
1,Global,0.166230-0.964592,7.710875e-16,-2.018961e-04,9.198985e-04,1.637901e-03,0.253215,—
1,Local 0.10,0.763810-0.828314,7.710875e-16,-1.320020e-04,8.763217e-04,1.620641e-03,0.086804,—
1,Local 0.03,0.761019-0.761965,7.710875e-16,-1.311386e-04,8.734906e-04,1.615843e-03,0.041698,—
2,Global,0.691913-0.003661,6.582395e-01,6.026855e-01,8.212064e-02,7.669268e-01,0.008049,—
2,Local 0.10,0.691822-0.000024,6.582395e-01,6.026122e-01,8.218702e-02,7.669862e-01,0.008254,—
2,Local 0.03,0.691785-0.000293,6.582395e-01,6.025692e-01,8.220623e-02,7.669816e-01,0.008227,—
3,Global,0.996398-0.949282-0.003798,-3.096391e-02,-5.617723e-02,7.579168e-02,9.540614e-02,0.420978,—
3,Local 0.10,0.899983-0.875577-0.472691,-3.096391e-02,-3.023653e-02,2.547957e-02,2.072260e-02,0.115882,—
3,Local 0.03,0.971468-0.934854-0.475738,-3.096391e-02,-3.047555e-02,1.816433e-02,5.853106e-03,0.050509,—
4,Global,0.395100-0.412426-0.408715-0.381967,4.098217e-01,8.188436e-02,2.366891e-01,5.552625e-01,0.059648,—


,Candidate search seconds
Function,
1,1.212
2,0.634
3,0.770
4,0.936
5,1.907
6,1.434
7,1.383
8,1.171


This cell compares global sampling with two local candidate pools around
the best observed input for each function. It generates 100,000 global
candidates and 10,000 candidates in each local neighbourhood. The local
boxes extend by up to 0.10 or 0.03 in each coordinate and are restricted
to the permitted domain.

Coordinates are rounded to six decimal places before prediction.
Duplicates within each pool and candidates within 1e-6 Euclidean distance
of an existing observation are excluded.

Each pool's proposal maximises the GP Upper Confidence Bound:
predicted mean + 2 × predictive standard deviation. The exploration
weight and sampling budgets remain unchanged from the previous round,
while the GP configurations reflect the current tuning experiment.

Auxiliary predictions for Functions 5 and 7 are reported at the
GP-selected candidates. They provide a separate comparison and do not
affect the UCB ranking.

*The highest UCB candidate is the best among the sampled candidates,
not a proven global optimum. “Global” and “local” describe candidate
generation, rather than directly identifying exploration or exploitation.
The reported GP uncertainty is model-dependent and is not guaranteed
to be calibrated. Search times exclude model fitting.*

The highest UCB proposals come from the global pool for Functions 1, 3,
6, 7 and 8, and from the Local 0.10 pool for Functions 2, 4 and 5.

For Function 5, both the GP and the auxiliary neural network predict an
output above the current best at the selected candidate. This agreement
supports investigating that region, but both predictions remain unverified.

For Function 7, the GP and linear regression continue to disagree
substantially. The auxiliary prediction is recorded as a diagnostic;
selection follows the predefined GP-UCB rule.

Function 2 has almost identical proposals and UCB scores across the
three pools. Its local candidate is selected by the numerical maximum,
rather than evidence of a meaningful advantage over the other proposals.

*For several functions, UCB selects a point whose predicted mean is below
the current best because its uncertainty bonus is larger. This is an
intentional exploration trade-off. A higher UCB is not a guarantee of
improvement, particularly where previous validation or prediction checks
have exposed model limitations.*

In [11]:
round_7_selected = {}
round_7_queries = {}
submission_rows = []

for function in range(1, 9):
    proposals = round_7_candidate_proposals[function]

    selected_search = max(
        proposals,
        key=lambda search: proposals[search]["ucb"]
    )
    proposal = proposals[selected_search]
    query = proposal["query"].copy()

    # Validate the exact submission coordinates.
    assert len(query) == updated_data[function]["X"].shape[1]
    assert np.isfinite(query).all()
    assert ((query >= 0) & (query < 1)).all()
    assert proposal["nearest_distance"] > 1e-6

    portal_query = "-".join(f"{value:.6f}" for value in query)

    round_7_queries[function] = query
    round_7_selected[function] = {
        **proposal,
        "query": query,
        "search": selected_search,
        "configuration": round_7_gp_config_names[function],
        "portal_query": portal_query,
        "beta": beta_round_7
    }

    submission_rows.append({
        "Function": function,
        "Selected search": selected_search,
        "Query": portal_query,
        "Predicted mean": proposal["mean"],
        "Predicted std": proposal["std"],
        "UCB": proposal["ucb"],
        "Auxiliary prediction": proposal["auxiliary_prediction"]
    })

round_7_submission = (
    pd.DataFrame(submission_rows).set_index("Function")
)

display(
    round_7_submission.style.format({
        "Predicted mean": "{:.6e}",
        "Predicted std": "{:.6e}",
        "UCB": "{:.6e}",
        "Auxiliary prediction": "{:.6e}"
    }, na_rep="—")
)

final_queries_table = (
    round_7_submission[["Query"]]
    .rename(columns={"Query": "Query for submission"})
)

display(final_queries_table)

# Plain-text version for convenient copying.
for function, row in final_queries_table.iterrows():
    print(f"Function {function}: {row['Query for submission']}")

,Selected search,Query,Predicted mean,Predicted std,UCB,Auxiliary prediction
Function,,,,,,
1,Global,0.166230-0.964592,-2.018961e-04,9.198985e-04,1.637901e-03,—
2,Local 0.10,0.691822-0.000024,6.026122e-01,8.218702e-02,7.669862e-01,—
3,Global,0.996398-0.949282-0.003798,-5.617723e-02,7.579168e-02,9.540614e-02,—
4,Local 0.10,0.401554-0.384590-0.372748-0.391438,2.147802e-01,2.415637e-01,6.979076e-01,—
5,Local 0.10,0.995096-0.996020-0.994275-0.997892,7.002326e+03,1.631969e+02,7.328720e+03,6.903573e+03
6,Global,0.476133-0.383212-0.776992-0.568645-0.009427,-2.809582e-01,1.319670e-01,-1.702420e-02,—
7,Global,0.048671-0.193593-0.957056-0.184874-0.553329-0.753430,2.129202e+00,1.734245e-01,2.476051e+00,9.154697e-01
8,Global,0.065894-0.388254-0.042724-0.117374-0.911203-0.566412-0.165832-0.270788,9.860398e+00,7.846440e-02,1.001733e+01,—


,Query for submission
Function,
1,0.166230-0.964592
2,0.691822-0.000024
3,0.996398-0.949282-0.003798
4,0.401554-0.384590-0.372748-0.391438
5,0.995096-0.996020-0.994275-0.997892
6,0.476133-0.383212-0.776992-0.568645-0.009427
7,0.048671-0.193593-0.957056-0.184874-0.553329-0...
8,0.065894-0.388254-0.042724-0.117374-0.911203-0...


Function 1: 0.166230-0.964592
Function 2: 0.691822-0.000024
Function 3: 0.996398-0.949282-0.003798
Function 4: 0.401554-0.384590-0.372748-0.391438
Function 5: 0.995096-0.996020-0.994275-0.997892
Function 6: 0.476133-0.383212-0.776992-0.568645-0.009427
Function 7: 0.048671-0.193593-0.957056-0.184874-0.553329-0.753430
Function 8: 0.065894-0.388254-0.042724-0.117374-0.911203-0.566412-0.165832-0.270788


This cell selects the highest-UCB proposal for each function across all
three candidate pools. It checks input dimensions, permitted coordinate
ranges and separation from previously evaluated observations.

The selected queries, GP predictions, predictive standard deviations,
auxiliary predictions and configuration names are retained in memory.
The final table provides the six-decimal, hyphen-separated strings
required by the portal.

*These are proposed Round 7 inputs, not observed results. Running this
cell does not submit them to the portal. Saving the executed notebook
preserves the displayed tables; the Python dictionaries remain available
only while the kernel session is active.*